In [ ]:
from pathlib import Path
import math
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
from skopt.utils import OptimizeResult
import sys

sys.path.append("../../src")
from mfdt.config_finder.ff_helpers import SerialOptimizeResult
from mfdt.config_finder.ff_figures import (
    plot_optimisation_process,
    _plot_mesh,
    _plot_trajectory,
)
import numpy as np
from mfdt.config_finder.ff_loss import (
    tau_loss,
    r_loss,
    r_tau_loss,
)

# Experiment 1&3

In [ ]:
EXPERIMENTS = [
    "exp_a",
    "exp_b",
    "exp_c",
    "exp_d",
    "exp_i",
]
NETWORK_TYPE = "bigreal"
NETWORK_NAME = "Freebase"

In [ ]:
for exp in EXPERIMENTS:
    serial_optim_result = SerialOptimizeResult.load(
        f"../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs/optim.pkl"
    )

    optim_result = OptimizeResult()
    optim_result.x_iters = serial_optim_result.x_iters
    optim_result.func_vals = serial_optim_result.func_vals

    optim_result.fun = serial_optim_result.fun
    optim_result.x = serial_optim_result.x

    save_path = Path(f"../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs/trajectory.png")

    plot_optimisation_process(
        result=optim_result,
        out_dir=save_path,
    )

# EXPERIMENT 2

In [ ]:
EXPERIMENTS = [
    "exp_e",
    "exp_f",
    "exp_g",
]
NETWORK_TYPE = "bigreal"
NETWORK_NAME = "Freebase"
N_STEPS = 30

In [ ]:
def plot_concat_optimisation_process(
    results: dict[str, OptimizeResult],
    out_dir: Path,
    exp: str,
) -> None:
    """Plot the optimisation process for a skopt result."""
    n_rows = math.ceil((len(results) + 1) / 2)
    fig, ax = plt.subplots(nrows=n_rows, ncols=2, figsize=(9, 4 * n_rows))
    fig.suptitle("Optimisation Results")

    ax[0, 0] = _plot_mesh(ax[0, 0], results[exp])
    title_name = " ".join(exp.upper().split("_"))
    ax[0, 0].set_title(f"Loss landscape & trajectory for {title_name}")

    remaining_exps = [e for e in EXPERIMENTS if e != exp]

    for i in range(ax.shape[0]):
        for j in range(ax.shape[1]):
            if i == 0 and j == 0:
                continue

            if i == 0:
                ax[i, j] = _plot_trajectory(ax[i, j], results[exp])
                title_name = " ".join(exp.upper().split("_"))
            else:
                e = remaining_exps[0]
                remaining_exps.remove(e)
                ax[i, j] = _plot_trajectory(ax[i, j], results[e])
                title_name = " ".join(e.upper().split("_"))

            ax[i, j].set_title(f"Convergence plot for {title_name}")

    assert len(remaining_exps) == 0

    fig.tight_layout()
    fig.savefig(out_dir, dpi=300)
    fig.savefig(out_dir.parent / "trajectory.png", dpi=300)
    fig.savefig(
        out_dir.parent / "trajectory.pdf",
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )


def load_otpim_Result(exp: str) -> OptimizeResult:
    serial_optim_result = SerialOptimizeResult.load(
        f"../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs/optim.pkl"
    )

    optim_result = OptimizeResult()
    optim_result.x_iters = serial_optim_result.x_iters
    optim_result.func_vals = serial_optim_result.func_vals

    optim_result.fun = serial_optim_result.fun
    optim_result.x = serial_optim_result.x

    return optim_result

In [ ]:
results = {exp: load_otpim_Result(exp) for exp in EXPERIMENTS}

for exp in EXPERIMENTS:
    save_path = Path(f"../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs/trajectory.png")

    plot_concat_optimisation_process(results=results, out_dir=save_path, exp=exp)

    result = results[exp]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax = _plot_trajectory(ax, result)
    fig.tight_layout()
    fig.savefig(save_path.parent / "only_trajectory.png", dpi=300)
    fig.savefig(
        save_path.parent / "only_trajectory.pdf",
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )

    fig, ax = plt.subplots(figsize=(5, 4))
    ax = _plot_mesh(ax, result)
    fig.tight_layout()
    fig.savefig(save_path.parent / "only_mesh.png", dpi=300)
    fig.savefig(
        save_path.parent / "only_mesh.pdf",
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )

In [ ]:
file_name = "trajectory_e_f_g"
for exp in EXPERIMENTS:
    save_path = Path(f"../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs")

    fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(14, 4))
    fig.suptitle("Optimisation Results")

    e = EXPERIMENTS[0]
    ax[0] = _plot_trajectory(ax[0], results[e])
    title_name = " ".join(e.upper().split("_"))
    ax[0].set_title(f"Convergence plot for {title_name}")

    e = EXPERIMENTS[1]
    ax[1] = _plot_trajectory(ax[1], results[e])
    title_name = " ".join(e.upper().split("_"))
    ax[1].set_title(f"Convergence plot for {title_name}")

    e = EXPERIMENTS[2]
    ax[2] = _plot_trajectory(ax[2], results[e])
    title_name = " ".join(e.upper().split("_"))
    ax[2].set_title(f"Convergence plot for {title_name}")

    fig.tight_layout()
    fig.savefig(save_path / f"{file_name}.png", dpi=300)
    fig.savefig(
        save_path / f"{file_name}.pdf",
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )

In [ ]:
file_name = "loss_r_tau_combined"

for exp in EXPERIMENTS:
    r = []
    r_best = []
    rb = np.inf

    tau = []
    tau_best = []
    tb = np.inf

    r_tau = []
    r_tau_best = []
    rtb = np.inf

    steps = []
    save_path = Path(f"../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs")
    a = np.load(save_path / "A.npy")
    b = np.load(save_path / "B.npy")

    for i in range(1, N_STEPS + 1):
        a_prime = np.load(save_path / str(i) / "A_primes.npy")
        b_prime = np.load(save_path / str(i) / "B_primes.npy")
        _r = r_loss(A=a, A_prime=a_prime)
        _tau = tau_loss(B=b, B_prime=b_prime)
        _r_tau = r_tau_loss(A=a, A_prime=a_prime, B=b, B_prime=b_prime)

        rb = rb if rb < _r else _r
        tb = tb if tb < _tau else _tau
        rtb = rtb if rtb < _r_tau else _r_tau

        r.append(_r)
        tau.append(_tau)
        r_tau.append(_r_tau)

        r_best.append(rb)
        tau_best.append(tb)
        r_tau_best.append(rtb)

        steps.append(i)

    r.append(None)
    tau.append(None)
    r_tau.append(None)

    r_best.append(None)
    tau_best.append(None)
    r_tau_best.append(None)

    steps.append(i + 1)

    df = pd.concat(
        [
            pd.DataFrame(
                {
                    "call": steps,
                    "loss": r,
                    "parameter": "r",
                    "label": "gp_current",
                }
            ),
            pd.DataFrame(
                {
                    "call": steps,
                    "loss": tau,
                    "parameter": "tau",
                    "label": "gp_current",
                }
            ),
            pd.DataFrame(
                {
                    "call": steps,
                    "loss": r_tau,
                    "parameter": "r+tau",
                    "label": "gp_current",
                }
            ),
            pd.DataFrame(
                {
                    "call": steps,
                    "loss": r_best,
                    "parameter": "r",
                    "label": "gp_best",
                }
            ),
            pd.DataFrame(
                {
                    "call": steps,
                    "loss": tau_best,
                    "parameter": "tau",
                    "label": "gp_best",
                }
            ),
            pd.DataFrame(
                {
                    "call": steps,
                    "loss": r_tau_best,
                    "parameter": "r+tau",
                    "label": "gp_best",
                }
            ),
        ]
    )

    fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(14, 4))

    sns.pointplot(
        data=df[(df["parameter"] == "r") & (df["label"] == "gp_current")],
        x="call",
        y="loss",
        linestyles="--",
        ax=ax[0],
        label="gp_current",
        linewidth=0.8,
        markers="",
    )
    sns.pointplot(
        data=df[(df["parameter"] == "r") & (df["label"] == "gp_best")],
        x="call",
        y="loss",
        ax=ax[0],
        color="#2C4A8A",
        label="gp_best",
    )

    sns.pointplot(
        data=df[(df["parameter"] == "tau") & (df["label"] == "gp_current")],
        x="call",
        y="loss",
        linestyles="--",
        ax=ax[1],
        label="gp_current",
        linewidth=0.8,
        markers="",
    )
    sns.pointplot(
        data=df[(df["parameter"] == "tau") & (df["label"] == "gp_best")],
        x="call",
        y="loss",
        ax=ax[1],
        color="#2C4A8A",
        label="gp_best",
    )

    sns.pointplot(
        data=df[(df["parameter"] == "r+tau") & (df["label"] == "gp_current")],
        x="call",
        y="loss",
        linestyles="--",
        ax=ax[2],
        label="gp_current",
        linewidth=0.8,
        markers="",
    )
    sns.pointplot(
        data=df[(df["parameter"] == "r+tau") & (df["label"] == "gp_best")],
        x="call",
        y="loss",
        ax=ax[2],
        color="#2C4A8A",
        label="gp_best",
    )

    ax[0].set_title(r"Convergence plot for $r$")
    ax[1].set_title(r"Convergence plot for $tau$")
    ax[2].set_title(r"Convergence plot for $r+tau$")

    for a in ax:
        a.set_ylabel(r"$\min f(x)$ after $n$ calls")
        a.set_xlabel("Number of calls $n$")
        a.grid(True, which="both", linestyle=":", linewidth=0.6, alpha=0.7)
        a.set_xlim(-1, 31)
        a.set_xticks([0, 5, 10, 15, 20, 25, 29])

    for a in ax:
        a.legend_.remove()

    handles, labels = ax[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        title="",
        loc="lower center",
        ncol=2,
        bbox_to_anchor=(0.5, -0.1),
    )

    plt.suptitle("Convergence plot of different parameters")
    plt.tight_layout()
    # plt.show()
    plt.savefig(
        save_path / f"{file_name}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.savefig(
        save_path / f"{file_name}.pdf",
        dpi=300,
        bbox_inches="tight",
    )